# 03 · Train on the ESCAPE benchmark

Same pipeline as notebook 01, but trained on the **ESCAPE** benchmark train split.

> **Setup:** place `escape_train.csv` (schema: `aa_seq, aa_len, AMP`) under
> `data/raw/escape/` and confirm the path in `config/default.yaml` → `escape.train_csv`.

In [ ]:
# --- bootstrap: works locally and on Google Colab ---
import sys, pathlib

REPO = "https://github.com/BioGavin/AMP-BERT.git"
try:
    import amp_bert  # already installed / on sys.path
except ModuleNotFoundError:
    root = pathlib.Path.cwd()
    root = root.parent if root.name == 'notebooks' else root
    src = root / 'src'
    if not (src / 'amp_bert').exists():
        # fresh Colab runtime: clone the repo
        import subprocess
        subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
        src = pathlib.Path('AMP-BERT') / 'src'
    sys.path.insert(0, str(src))

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
from transformers import Trainer
from amp_bert import (AmpDataset, load_dataset, compute_metrics,
                      model_init, build_training_args)
from amp_bert.config import RAW_DIR, MODELS_DIR, RESULTS_DIR

In [ ]:
ESCAPE_TRAIN_CSV = RAW_DIR / 'escape' / 'escape_train.csv'
assert ESCAPE_TRAIN_CSV.exists(), (
    f'ESCAPE train data not found at {ESCAPE_TRAIN_CSV}. '
    'Add it (see data/README.md).')

## Load ESCAPE train split

In [ ]:
df = load_dataset(ESCAPE_TRAIN_CSV, shuffle=True, random_state=0)
print(df.shape)
df.head()

In [ ]:
train_dataset = AmpDataset(df, max_len=200)

## Train & save

In [ ]:
out_dir = str(RESULTS_DIR / 'escape_train')
training_args = build_training_args(out_dir, num_train_epochs=15, seed=0)
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
save_path = MODELS_DIR / 'amp_bert_escape'
trainer.save_model(str(save_path))
print('saved to', save_path)